# Lab 1 — A tiny V-JEPA from scratch

We will preserve the essential classical V-JEPA learning graph while making every tensor small enough to inspect:

1. generate moving-square videos;
2. convert video cubes into tubelet tokens;
3. hide one spatiotemporal block;
4. predict target-encoder embeddings from visible context;
5. update the target encoder with an exponential moving average (EMA);
6. check feature spread and effective rank alongside loss.

> **Scope.** This is a teaching implementation, not a numerical reproduction of Meta's V-JEPA recipe.


In [ ]:
# imports and reproducibility
import copy
import math
import random
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FAST_MODE = True
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)


## 1. Video generation

Let's create a synthetic video - a sequence of actions

In [ ]:
BATCH_SIZE=1 # single sequence
FRAMES=8
SIZE=32 # frame resolution (32x32)
SQUARE=5 # moving object size (a square 5x5)
NOISE=0.03 # gaussian noise added to pixels
DEVICE="cuda"

def make_videos(batch_size, frames, size, square, noise, device):
    videos = torch.zeros(batch_size, 1, frames, size, size, device=device)
    pos = torch.randint(2, size - square - 2, (batch_size, 2), device=device)
    speed_choices = torch.tensor([-2, -1, 1, 2], device=device)
    vel = speed_choices[torch.randint(0, len(speed_choices), (batch_size, 2), device=device)]

    for t in range(frames):
        for b in range(batch_size):
            x, y = pos[b].tolist()
            videos[b, 0, t, y:y + square, x:x + square] = 1.0
        proposed = pos + vel
        hit_low = proposed < 0
        hit_high = proposed > (size - square)
        vel = torch.where(hit_low | hit_high, -vel, vel)
        pos = (pos + vel).clamp(0, size - square)

    videos = (videos + noise * torch.randn_like(videos)).clamp(0, 1)
    return videos

# Now use the method to create a video
demo_video = make_videos(BATCH_SIZE, FRAMES, SIZE, SQUARE, NOISE, DEVICE).cpu()
print("Video shape [B,C,T,H,W]:", tuple(demo_video.shape))


Now let's inspect all frames

In [ ]:
fig, axes = plt.subplots(1, demo_video.shape[2], figsize=(14, 2))
for t, ax in enumerate(axes):
    ax.imshow(demo_video[0, 0, t], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"t={t}")
    ax.axis("off")
plt.suptitle("Synthetic video")
plt.tight_layout()
plt.show()


## 2. Tubelet tokenization

A 3D convolution with kernel and stride $(\tau,P,P)$ maps a clip into a regular token grid:

$$N=\frac{T}{\tau}\frac{H}{P}\frac{W}{P}.$$

Here, $T=8$, $\tau=2$, $H=W=32$, and $P=8$, so $N=4\times4\times4=64$ tokens.


Now let's create the video encoder and compute tubelet tokenization

In [ ]:
class VideoEncoder(nn.Module):
    def __init__(
        self,
        dim=64,        # D: embedding dimension of each tubelet token
        depth=2,       # Number of Transformer encoder blocks
        heads=4,       # Number of attention heads per Transformer block
        frames=8,      # T: number of input video frames
        size=32,       # H = W: spatial resolution of each frame (32x32)
        tubelet=2,     # tau: number of consecutive frames covered by one tubelet
        patch=8        # P: spatial size of each tubelet (8x8 pixels)
    ):
        super().__init__()


        # define tubelet embedding (3d conv)
        self.proj = nn.Conv3d(
            in_channels=1,
            out_channels=dim,
            kernel_size=(tubelet, patch, patch),
            stride=(tubelet, patch, patch)
        )


        # define tubelet grid
        self.grid = (
            frames // tubelet,
            size // patch,
            size // patch
        )

        # define total number of tokens based on the grid
        self.num_tokens = math.prod(self.grid)


        # define positional embeddings
        self.pos = nn.Parameter(
            torch.randn(1, self.num_tokens, dim) * 0.02
        )


        # define one transformer encoder block
        layer = nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=heads,
            dim_feedforward=dim * 3,
            dropout=0.0,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )


        # stack multiple transformer blocks
        self.blocks = nn.TransformerEncoder(
            layer,
            num_layers=depth
        )

        # final normalization applied to the encoder output
        self.norm = nn.LayerNorm(dim)


    def tokenize(self, video):

        # compute video -> tubelet embeddings
        x = self.proj(video)

        # flatten the spatiotemporal grid
        x = x.flatten(2)

        # transpose to transformer format
        return x.transpose(1, 2)


    def forward(self, video, indices=None):

       # start by tokenizing the video
        x = self.tokenize(video)

        # add positional information
        # each token recieves its corresponding learned
        # spatiotemporal positional embedding:
        # token_i = content_i + position_i
        # this allows the transformer to distinguish tokens
        # coming from different times and spatial locations
        x = x + self.pos


        # optionally select only some tokens
        # for example, we may start with: [B, 64, D]
        # and keep only context / visible tokens: [B, N_context, D]
        if indices is not None:
            x = x[:, indices]


        # apply transformers encoding
        # self-attention allows each visible token to use
        # information from the other visible tubelets
        # Before the Transformer:
        # each token mainly represents its own local tubelet
        # After the Transformer:
        # each token is contextualized using information from
        # other tubelets across space and time
        x = self.blocks(x)
        x = self.norm(x)
        return x

In [ ]:
# testing video encoder features
# tokenization

scratch_encoder = VideoEncoder().to(DEVICE)

with torch.no_grad():

    # only run tokenization here!
    # we are NOT passing the tokens through the Transformer yet
    tokens = scratch_encoder.tokenize(demo_video.to(DEVICE))

print("Tubelet grid [T',H',W']:",scratch_encoder.grid)
print("Tokens [B,N,D]:",tuple(tokens.shape))

## 3. One spatiotemporal target block

The real V-JEPA recipe samples several masks with different spatial and temporal scales. For transparency, the lab uses one rectangular target block shared by a batch.

In [ ]:
# sample target indices and use every other token as context
def token_id(t, h, w, grid):
    _, gh, gw = grid
    return t * gh * gw + h * gw + w


def block_mask(grid, block=(2, 2, 2), seed=0):
    gt, gh, gw = grid
    bt, bh, bw = block
    rng = random.Random(seed)
    t0 = rng.randint(0, gt - bt)
    h0 = rng.randint(0, gh - bh)
    w0 = rng.randint(0, gw - bw)
    targets = [token_id(t, h, w, grid)
               for t in range(t0, t0 + bt)
               for h in range(h0, h0 + bh)
               for w in range(w0, w0 + bw)]
    context = [i for i in range(math.prod(grid)) if i not in set(targets)]
    return torch.tensor(context), torch.tensor(targets)


context_idx, target_idx = block_mask(scratch_encoder.grid, seed=SEED)
print(f"Context tokens: {len(context_idx)} | target tokens: {len(target_idx)}")
print("Target indices:", target_idx.tolist())

In [ ]:
# show the target location at each tubelet time
mask_grid = torch.zeros(scratch_encoder.grid)
for idx in target_idx:
    t = idx // (scratch_encoder.grid[1] * scratch_encoder.grid[2])
    rem = idx % (scratch_encoder.grid[1] * scratch_encoder.grid[2])
    h, w = rem // scratch_encoder.grid[2], rem % scratch_encoder.grid[2]
    mask_grid[t, h, w] = 1

fig, axes = plt.subplots(1, scratch_encoder.grid[0], figsize=(9, 2.2))
for t, ax in enumerate(axes):
    ax.imshow(mask_grid[t], cmap="Purples", vmin=0, vmax=1)
    ax.set_title(f"tube time {t}")
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("PURPLE = prediction target")
plt.tight_layout()
plt.show()


### 4. Predictor and EMA target encoder

The online encoder sees only context indices. The predictor places those encoded tokens into a full sequence, fills missing positions with a learned mask token, and predicts embeddings at target indices. The target encoder sees the full clip but receives no gradient.

In [ ]:
class Predictor(nn.Module):
    def __init__(
        self,
        encoder_dim=64,    # D_enc: dimension of the encoder representations
        predictor_dim=48,  # D_pred: internal dimension used by the predictor
        depth=2,           # Number of Transformer blocks in the predictor
        heads=4,           # Number of self-attention heads
        num_tokens=64      # N: total number of tubelet positions in the video
    ):
        super().__init__()

        # project encoder features into the predictor space
        self.in_proj = nn.Linear(
            encoder_dim,
            predictor_dim
        )

        # define the learnable mask token
        # For positions that are NOT provided by the context encoder,
        # the predictor starts from the same learned mask token
        # Later, this token is repeated across all N token positions
        # IMPORTANT:
        # This does NOT contain the target representation
        # It is only a learned placeholder saying:
        # "the representation at this position must be predicted"
        self.mask_token = nn.Parameter(
            torch.zeros(1, 1, predictor_dim)
        )

        # define positional embeddings
        # the predictor must know WHICH position each token corresponds to
        self.pos = nn.Parameter(
            torch.randn(1, num_tokens, predictor_dim) * 0.02
        )

        # define one predictor transformer block
        layer = nn.TransformerEncoderLayer(
            d_model=predictor_dim,
            nhead=heads,
            dim_feedforward=predictor_dim * 3,
            dropout=0.0,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        # stacked transformer blocks
        self.blocks = nn.TransformerEncoder(
            layer,
            num_layers=depth
        )

        # define projecting predictions back to the encoder space
        self.out_proj = nn.Linear(
            predictor_dim,
            encoder_dim
        )


    def forward(
        self,
        context_z,        # Encoded visible/context tokens: [B, N_context, D_enc]
        context_indices,  # Positions of the context tokens in the original grid
        target_indices    # Positions that we want to predict
    ):

        batch = context_z.shape[0]

        # create a full sequence of mask tokens
        x = self.mask_token.expand(
            batch,
            self.pos.shape[1],
            -1
        ).clone()

        # insert the real context representations
        x[:, context_indices] = self.in_proj(context_z)

        # add positional information
        x = x + self.pos

        # predict through self-attention
        x = self.blocks(x)

        # keep only the target positions
        # We do not need predictions for every token.
        # We only select the positions specified by target_indices
        x = x[:, target_indices]

        # project back to the encoder representation space
        return self.out_proj(x)


# define ema update for the target encoder
@torch.no_grad()
def ema_update(target, online, momentum=0.99):

    for target_param, online_param in zip(target.parameters(),online.parameters()):
        target_param.mul_(momentum).add_(online_param,alpha=1 - momentum)


### 5. Train briefly

The loss is an embedding prediction error:

$$\mathcal L=\left\|g_\phi(f_\theta(x_{ctx}),m)-\operatorname{sg}(f_{\bar\theta}(x)_{tgt})\right\|_1.$$

This cell intentionally runs for seconds/minutes, not to convergence.

In [ ]:
# initialize the complete learning graph
torch.manual_seed(SEED)
online = VideoEncoder().to(DEVICE)
target = copy.deepcopy(online).to(DEVICE).eval()

for p in target.parameters():
    p.requires_grad_(False)
predictor = Predictor(num_tokens=online.num_tokens).to(DEVICE)

optimizer = torch.optim.AdamW(
    list(online.parameters()) + list(predictor.parameters()),
    lr=3e-4, weight_decay=0.04,
)

context_idx = context_idx.to(DEVICE)
target_idx = target_idx.to(DEVICE)


In [ ]:
# training loop

STEPS = 300
BATCH = 24 if DEVICE == "cuda" else 12
PRINT_EVERY = 25

# store the loss from every optimization step.
losses = []
start = time.time()

# the online/context encoder and predictor are optimized with gradients
online.train()
predictor.train()

# the target encoder is NOT optimized with gradients
# tts weights are updated only through EMA
target.eval()

# start training loop
for step in range(STEPS):
    video = make_videos(
        BATCH,
        FRAMES,
        SIZE,
        SQUARE,
        NOISE,
        DEVICE
    )

   # call context encoder
    context_z = online(
        video,
        context_idx
    )

    # predict target representations
    predicted = predictor(
        context_z,
        context_idx,
        target_idx
    )

    # compute target representations
    with torch.no_grad():
        target_z = target(video)[:, target_idx]

    # define jepa loss
    loss = F.smooth_l1_loss(
        predicted,
        target_z
    )

    # clear old gradients
    optimizer.zero_grad(set_to_none=True)

    # and let's go for backprop!
    loss.backward()

    # some gradient clipping to prevent very large gradients from destabilizing training
    torch.nn.utils.clip_grad_norm_(
        list(online.parameters()) +
        list(predictor.parameters()),
        max_norm=1.0
    )

    # update online encoder + predictor
    optimizer.step()

    # update target encoder with ema
    # 10. UPDATE TARGET ENCODER WITH EMA
    ema_update(
        target,
        online,
        momentum=0.99
    )

    # store losses!
    losses.append(loss.item())

   # just some progress printing...
    if (
        step == 0
        or (step + 1) % PRINT_EVERY == 0
        or step == STEPS - 1
    ):
        print(
            f"step {step + 1:03d}/{STEPS} "
            f"| loss {loss.item():.6f}"
        )



# just some training time...
elapsed = time.time() - start
print(f"\nElapsed: {elapsed:.1f}s")
print(f"Initial loss: {losses[0]:.6f}")
print(f"Final loss:   {losses[-1]:.6f}")


# plots
plt.figure(figsize=(8, 4))

plt.plot(
    range(1, STEPS + 1),
    losses,
    label="Training loss"
)

plt.xlabel("Training step")
plt.ylabel("Smooth L1 loss")
plt.title("JEPA latent prediction loss")
plt.grid(alpha=0.3)
plt.legend()

plt.show()

### 6. Collapse diagnostics

We measure per-dimension feature spread and covariance effective rank. These catch an obvious constant representation, but they do **not** prove semantic quality.

In [ ]:
@torch.no_grad()
def representation_diagnostics(encoder, videos):
    encoder.eval()
    pooled = encoder(videos).mean(dim=1)
    centered = pooled - pooled.mean(dim=0, keepdim=True)
    feature_std = pooled.std(dim=0).mean().item()
    covariance = centered.T @ centered / max(1, len(centered) - 1)
    eig = torch.linalg.eigvalsh(covariance).clamp_min(1e-12)
    probabilities = eig / eig.sum()
    effective_rank = torch.exp(-(probabilities * probabilities.log()).sum()).item()
    return feature_std, effective_rank, pooled


probe_videos = make_videos(
        128,
        FRAMES,
        SIZE,
        SQUARE,
        NOISE,
        DEVICE
    )
std, rank, pooled = representation_diagnostics(online, probe_videos)
print(f"Mean feature std: {std:.4f}")
print(f"Covariance effective rank: {rank:.1f} / {pooled.shape[-1]}")
assert torch.isfinite(pooled).all(), "Non-finite representation"


The collapse diagnostics provide additional information about the learned representation. The non-zero mean feature standard deviation (**0.0257**) shows that the encoder does not produce exactly the same representation for every input. In addition, the covariance effective rank is **14.6 out of 64 dimensions**, indicating that the variation in the learned embeddings is distributed across multiple directions, although considerably fewer than the full 64-dimensional space. For this small synthetic experiment, these measurements suggest that the representation is not trivially constant, while also showing substantial redundancy in the learned feature space.

